# 🫁 Pneumonia Detection from Chest X-Rays — Deep Learning + Grad-CAM
### Transfer learning (EfficientNetB0) · Explainable AI · Streamlit demo

This Colab notebook trains a CNN to classify chest X-rays as **NORMAL** vs **PNEUMONIA**, evaluates it, and uses
**Grad-CAM** to visualize *where* the model looks. It saves the trained model and all figures to `outputs/` for the
GitHub repo and the Streamlit app.

**▶ Before running:** `Runtime → Change runtime type → GPU` (T4 is free).

**Dataset:** [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia)
— 5,863 images. You'll need a free Kaggle API token (`kaggle.json`).

## 1. Setup & GPU check

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow.keras import layers, models
print('TensorFlow', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'NONE — set Runtime → GPU!')
os.makedirs('outputs/figures', exist_ok=True)
plt.rcParams['figure.dpi']=110

## 2. Download the dataset (Kaggle)
Get `kaggle.json` from your Kaggle account (Account → Create New API Token), then run the cell and upload it.

In [ ]:
from google.colab import files
print('Upload your kaggle.json:'); files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -q
!unzip -q -o chest-xray-pneumonia.zip -d data
# locate the folder that contains train/ val/ test/
import glob
ROOT = next(os.path.dirname(p) for p in glob.glob('data/**/train', recursive=True))
print('Data root:', ROOT)

## 3. Explore the data

In [ ]:
import collections
for split in ['train','val','test']:
    counts={c:len(os.listdir(f'{ROOT}/{split}/{c}')) for c in ['NORMAL','PNEUMONIA']}
    print(f'{split:5s}: {counts}  (total {sum(counts.values())})')

# show a few samples
fig,ax=plt.subplots(2,4,figsize=(12,6))
for j,cls in enumerate(['NORMAL','PNEUMONIA']):
    fs=os.listdir(f'{ROOT}/train/{cls}')[:4]
    for i,f in enumerate(fs):
        im=tf.keras.utils.load_img(f'{ROOT}/train/{cls}/{f}'); ax[j,i].imshow(im,cmap='gray'); ax[j,i].set_title(cls); ax[j,i].axis('off')
plt.tight_layout(); plt.savefig('outputs/figures/sample_xrays.png',bbox_inches='tight'); plt.show()

## 4. Data pipelines

In [ ]:
IMG=224; BATCH=32
train_ds=tf.keras.utils.image_dataset_from_directory(f'{ROOT}/train', labels='inferred', label_mode='binary',
    class_names=['NORMAL','PNEUMONIA'], image_size=(IMG,IMG), batch_size=BATCH, validation_split=0.2, subset='training', seed=42)
val_ds=tf.keras.utils.image_dataset_from_directory(f'{ROOT}/train', labels='inferred', label_mode='binary',
    class_names=['NORMAL','PNEUMONIA'], image_size=(IMG,IMG), batch_size=BATCH, validation_split=0.2, subset='validation', seed=42)
test_ds=tf.keras.utils.image_dataset_from_directory(f'{ROOT}/test', labels='inferred', label_mode='binary',
    class_names=['NORMAL','PNEUMONIA'], image_size=(IMG,IMG), batch_size=BATCH, shuffle=False)

# class weights (the data is imbalanced toward PNEUMONIA)
import numpy as np
n_norm=len(os.listdir(f'{ROOT}/train/NORMAL')); n_pneu=len(os.listdir(f'{ROOT}/train/PNEUMONIA')); tot=n_norm+n_pneu
class_weight={0: tot/(2*n_norm), 1: tot/(2*n_pneu)}
print('class_weight:', class_weight)

# augmentation applied to the training set only; EfficientNet expects pixels in [0,255] (normalization is built in)
data_aug=tf.keras.Sequential([layers.RandomFlip('horizontal'), layers.RandomRotation(0.08), layers.RandomZoom(0.1)])
AUTO=tf.data.AUTOTUNE
train_ds=train_ds.map(lambda x,y:(data_aug(x,training=True),y),num_parallel_calls=AUTO).prefetch(AUTO)
val_ds=val_ds.cache().prefetch(AUTO); test_ds=test_ds.cache().prefetch(AUTO)

## 5. Build the model (EfficientNetB0 transfer learning)

In [ ]:
inputs=tf.keras.Input((IMG,IMG,3))
base=tf.keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_tensor=inputs)
base.trainable=False
x=layers.GlobalAveragePooling2D()(base.output)
x=layers.Dropout(0.3)(x)
outputs=layers.Dense(1, activation='sigmoid')(x)
model=tf.keras.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc'), tf.keras.metrics.Precision(name='prec'), tf.keras.metrics.Recall(name='rec')])
model.summary()

## 6. Train the classifier head

In [ ]:
cbs=[tf.keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=4, restore_best_weights=True),
     tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2)]
h1=model.fit(train_ds, validation_data=val_ds, epochs=12, class_weight=class_weight, callbacks=cbs)

## 7. Fine-tune the top of the backbone

In [ ]:
base.trainable=True
for l in base.layers[:-30]: l.trainable=False
for l in base.layers:        # keep BatchNorm frozen during fine-tuning
    if isinstance(l, layers.BatchNormalization): l.trainable=False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc'), tf.keras.metrics.Precision(name='prec'), tf.keras.metrics.Recall(name='rec')])
h2=model.fit(train_ds, validation_data=val_ds, epochs=8, class_weight=class_weight, callbacks=cbs)

## 8. Training curves

In [ ]:
hist={k: h1.history[k]+h2.history.get(k,[]) for k in h1.history}
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].plot(hist['accuracy'],label='train'); ax[0].plot(hist['val_accuracy'],label='val'); ax[0].set_title('Accuracy'); ax[0].legend()
ax[1].plot(hist['loss'],label='train'); ax[1].plot(hist['val_loss'],label='val'); ax[1].set_title('Loss'); ax[1].legend()
plt.tight_layout(); plt.savefig('outputs/figures/training_history.png',bbox_inches='tight'); plt.show()

## 9. Evaluate on the test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import seaborn as sns
y_true=np.concatenate([y.numpy() for _,y in test_ds]).ravel().astype(int)
y_prob=model.predict(test_ds).ravel()
y_pred=(y_prob>=0.5).astype(int)
print(classification_report(y_true,y_pred,target_names=['NORMAL','PNEUMONIA'],digits=3))

cm=confusion_matrix(y_true,y_pred)
plt.figure(figsize=(5,4)); sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=['NORMAL','PNEUMONIA'],yticklabels=['NORMAL','PNEUMONIA'])
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix'); plt.tight_layout()
plt.savefig('outputs/figures/confusion_matrix.png',bbox_inches='tight'); plt.show()

fpr,tpr,_=roc_curve(y_true,y_prob); roc_auc=auc(fpr,tpr)
plt.figure(figsize=(5,4)); plt.plot(fpr,tpr,label=f'AUC = {roc_auc:.3f}',color='#d62728'); plt.plot([0,1],[0,1],'--',color='gray')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('ROC Curve'); plt.legend(); plt.tight_layout()
plt.savefig('outputs/figures/roc_curve.png',bbox_inches='tight'); plt.show()
print('Test ROC-AUC:', round(roc_auc,4))

## 10. Grad-CAM — what is the model looking at?

In [ ]:
last_conv=[l.name for l in model.layers if isinstance(l, layers.Conv2D)][-1]
print('Last conv layer:', last_conv)

def gradcam(img_array, model, last_conv_name):
    grad_model=tf.keras.Model(model.inputs, [model.get_layer(last_conv_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array); loss = preds[:,0]
    grads=tape.gradient(loss, conv_out)
    pooled=tf.reduce_mean(grads, axis=(0,1,2))
    hm=tf.squeeze(conv_out[0] @ pooled[...,tf.newaxis])
    hm=tf.maximum(hm,0)/(tf.reduce_max(hm)+1e-8)
    return hm.numpy()

import matplotlib.cm as cm_
def overlay(path):
    img=tf.keras.utils.load_img(path,target_size=(IMG,IMG)); arr=tf.keras.utils.img_to_array(img)
    hm=gradcam(arr[None,...], model, last_conv)
    hm=tf.image.resize(hm[...,None],(IMG,IMG)).numpy().squeeze()
    heat=cm_.jet(hm)[...,:3]*255
    blend=(0.6*arr+0.4*heat).astype('uint8'); prob=float(model.predict(arr[None,...],verbose=0)[0,0])
    return img, blend, prob

samples=[(f'{ROOT}/test/PNEUMONIA/'+os.listdir(f'{ROOT}/test/PNEUMONIA')[0],'PNEUMONIA'),
         (f'{ROOT}/test/NORMAL/'+os.listdir(f'{ROOT}/test/NORMAL')[0],'NORMAL')]
fig,ax=plt.subplots(2,2,figsize=(9,9))
for r,(p,lbl) in enumerate(samples):
    img,blend,prob=overlay(p)
    ax[r,0].imshow(img); ax[r,0].set_title(f'Original — {lbl}'); ax[r,0].axis('off')
    ax[r,1].imshow(blend); ax[r,1].set_title(f'Grad-CAM — P(pneumonia)={prob:.2f}'); ax[r,1].axis('off')
plt.tight_layout(); plt.savefig('outputs/figures/gradcam_examples.png',bbox_inches='tight'); plt.show()

## 11. Save the model & download everything

In [ ]:
model.save('outputs/pneumonia_efficientnet.keras')
with open('outputs/metrics.txt','w') as f:
    f.write(classification_report(y_true,y_pred,target_names=['NORMAL','PNEUMONIA'],digits=3))
    f.write(f'\nTest ROC-AUC: {roc_auc:.4f}\n')
!cd outputs && zip -q -r ../pneumonia_outputs.zip . && echo "zipped"
from google.colab import files; files.download('pneumonia_outputs.zip')
print('Done — unzip into your repo (model + figures) and push.')

## 12. Next steps
1. Unzip `pneumonia_outputs.zip` → put `pneumonia_efficientnet.keras` in `app/` and the `figures/*.png` in `assets/`.
2. Run the **Streamlit demo** locally: `streamlit run app/app.py`.
3. Commit & push. See `STEPS.md` in the repo for the exact commands.

*Disclaimer: this is an educational project, not a medical device. Do not use for diagnosis.*